# EAAI-26-12471 — Step 1: Dataset audit (leakage check)
Run on Kaggle with: Add Input -> "paultimothymooney/blood-cells", Accelerator -> GPU.
Outputs go to /kaggle/working. Send back: the printed SUMMARY + the files listed at the end.

In [ ]:
import os, re, glob, json
import numpy as np, pandas as pd
ROOT = os.environ.get("AUDIT_ROOT", "/kaggle/input")
OUT = os.environ.get("AUDIT_OUT", "/kaggle/working")
WEIGHTS = os.environ.get("AUDIT_WEIGHTS", "imagenet")
WEIGHTS = None if WEIGHTS == "None" else WEIGHTS
os.makedirs(OUT, exist_ok=True)
imgs = sorted(p for p in glob.glob(f"{ROOT}/**/*", recursive=True)
              if p.lower().endswith((".jpg", ".jpeg", ".png")))
print("Total image files found:", len(imgs))
assert imgs, "No images found - check that the dataset is attached as input."

## 1. Folder structure and class counts

In [ ]:
rows = []
for p in imgs:
    rel = os.path.relpath(p, ROOT).split(os.sep)
    rows.append(dict(path=p, file=rel[-1],
                     cls=rel[-2] if len(rel) >= 2 else "",
                     split=rel[-3] if len(rel) >= 3 else "",
                     src="/".join(rel[:-3]) if len(rel) > 3 else ""))
df = pd.DataFrame(rows)
struct = df.groupby(["src", "split", "cls"]).size().rename("n").reset_index()
print(struct.to_string(index=False))
struct.to_csv(f"{OUT}/audit_structure.csv", index=False)

## 2. File-name patterns (do names encode the source image?)

In [ ]:
df["pattern"] = df.file.str.replace(r"\d+", "#", regex=True)
print(df.groupby(["split", "pattern"]).size().sort_values(ascending=False).head(20).to_string())
tok = df.file.str.extract(r"^_?(\d+)_(\d+)")
df["tokA"], df["tokB"] = tok[0], tok[1]
for t in ["tokA", "tokB"]:
    sub = df.dropna(subset=[t])
    if sub.empty:
        continue
    print(f"\n{t}: unique values per split/class")
    print(sub.groupby(["split", "cls"])[t].nunique().to_string())
    if {"TRAIN", "TEST"} <= set(sub.split):
        a = set(sub[sub.split == "TRAIN"][t]); b = set(sub[sub.split == "TEST"][t])
        print(f"{t}: TRAIN unique={len(a)}, TEST unique={len(b)}, shared={len(a & b)}")

## 3. Labels of the original (pre-augmentation) images

In [ ]:
for lp in glob.glob(f"{ROOT}/**/labels.csv", recursive=True):
    lab = pd.read_csv(lp)
    print("\n", lp, "| columns:", list(lab.columns), "| rows:", len(lab))
    for c in lab.columns:
        if not pd.api.types.is_numeric_dtype(lab[c]) and lab[c].nunique() < 30:
            print(lab[c].value_counts(dropna=False).to_string())

## 4. Near-duplicate detection with rotation/flip-invariant embeddings

In [ ]:
import tensorflow as tf
aug = df[df.split.isin(["TRAIN", "TEST"])].reset_index(drop=True)
print("Images used for duplicate analysis:", len(aug), aug.split.value_counts().to_dict())
base = tf.keras.applications.EfficientNetB0(include_top=False, pooling="avg", weights=WEIGHTS)
pre = tf.keras.applications.efficientnet.preprocess_input

def load(p):
    x = tf.image.decode_jpeg(tf.io.read_file(p), channels=3)
    return tf.image.resize(x, (224, 224))

def embed(paths, bs=int(os.environ.get("AUDIT_BS", 64))):
    ds = (tf.data.Dataset.from_tensor_slices(paths)
          .map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(bs).prefetch(2))
    out = []
    for b in ds:
        acc = 0.0
        for flip in (False, True):
            bb = tf.image.flip_left_right(b) if flip else b
            for k in range(4):  # 8 variants = full dihedral group D4
                acc += base(pre(tf.image.rot90(bb, k)), training=False)
        out.append((acc / 8.0).numpy())
    E = np.concatenate(out).astype(np.float32)
    return E / np.linalg.norm(E, axis=1, keepdims=True)

E = embed(aug.path.tolist())
np.save(f"{OUT}/audit_embeddings.npy", E)
print("Embeddings:", E.shape)

In [ ]:
is_tr = (aug.split == "TRAIN").values
Etr, Ete = E[is_tr], E[~is_tr]
ytr, yte = aug.cls.values[is_tr], aug.cls.values[~is_tr]
nn_sim = np.empty(len(Ete), np.float32); nn_idx = np.empty(len(Ete), np.int64)
for i in range(0, len(Ete), 1024):
    S = Ete[i:i + 1024] @ Etr.T
    nn_idx[i:i + 1024] = S.argmax(1)
    nn_sim[i:i + 1024] = S.max(1)
same_cls = (ytr[nn_idx] == yte)
qs = [0.01, 0.05, 0.25, 0.5, 0.75, 0.95]
print("TEST -> nearest TRAIN cosine quantiles:", dict(zip(qs, np.round(np.quantile(nn_sim, qs), 4))))
for thr in [0.90, 0.93, 0.95, 0.97, 0.99]:
    m = nn_sim >= thr
    print(f"thr {thr}: {m.mean()*100:5.1f}% of TEST have a TRAIN neighbour; label agreement among them = "
          f"{(same_cls[m].mean()*100 if m.any() else float('nan')):.1f}%")
print(f"1-NN label agreement overall (a pure lookup 'classifier'): {same_cls.mean()*100:.2f}%")
pd.DataFrame(dict(test=aug.path.values[~is_tr], test_cls=yte, nn_train=aug.path.values[is_tr][nn_idx],
                  nn_cls=ytr[nn_idx], cos=nn_sim)).to_csv(f"{OUT}/audit_test_nn.csv", index=False)

## 5. Group structure: how many distinct source cells? Do groups span TRAIN and TEST?

In [ ]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

def groups(E, thr, chunk=1024, max_edges=50_000_000):
    rr, cc, n_e = [], [], 0
    for i in range(0, len(E), chunk):
        r, c = np.nonzero(E[i:i + chunk] @ E.T >= thr)
        r = r + i; k = r < c
        rr.append(r[k]); cc.append(c[k]); n_e += int(k.sum())
        if n_e > max_edges:
            return None, n_e
    r, c = np.concatenate(rr), np.concatenate(cc)
    A = coo_matrix((np.ones(len(r), np.int8), (r, c)), shape=(len(E), len(E)))
    return connected_components(A, directed=False)[1], n_e

group_summary = []
for thr in [0.93, 0.95, 0.97, 0.99]:
    g, n_e = groups(E, thr)
    if g is None:
        print(f"thr {thr}: skipped (>{n_e} edges)"); continue
    sp = pd.DataFrame(dict(g=g, split=aug.split.values, cls=aug.cls.values))
    per = sp.groupby("g").agg(n=("split", "size"), splits=("split", "nunique"), classes=("cls", "nunique"))
    test_in_shared = sp[sp.g.isin(per.index[per.splits > 1]) & (sp.split == "TEST")].shape[0]
    rec = dict(thr=thr, n_groups=len(per), groups_spanning_train_test=int((per.splits > 1).sum()),
               pct_test_in_shared_groups=round(100 * test_in_shared / (~is_tr).sum(), 2),
               mixed_label_groups=int((per.classes > 1).sum()), largest_group=int(per.n.max()))
    group_summary.append(rec); print(rec)
    aug[f"group_{thr}"] = g
pd.DataFrame(group_summary).to_csv(f"{OUT}/audit_groups_summary.csv", index=False)
aug.drop(columns=["pattern"], errors="ignore").to_csv(f"{OUT}/audit_groups.csv", index=False)

## 6. Visual check: most similar and median TEST/TRAIN pairs

In [ ]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
def pair_grid(order, title, fname, k=8):
    fig, ax = plt.subplots(2, k, figsize=(2.2 * k, 5))
    for j, i in enumerate(order[:k]):
        for r, (p, lab) in enumerate([(aug.path.values[~is_tr][i], "TEST " + yte[i]),
                                      (aug.path.values[is_tr][nn_idx[i]], "TRAIN " + ytr[nn_idx[i]])]):
            ax[r, j].imshow(Image.open(p).convert("RGB")); ax[r, j].axis("off")
            ax[r, j].set_title(f"{lab[:14]}\n{nn_sim[i]:.3f}" if r == 0 else lab[:14], fontsize=7)
    fig.suptitle(title); fig.tight_layout(); fig.savefig(f"{OUT}/{fname}", dpi=110); plt.close(fig)
o = np.argsort(-nn_sim)
pair_grid(o, "Most similar TEST/TRAIN pairs", "audit_pairs_top.png")
pair_grid(o[len(o) // 2:], "Median-similarity TEST/TRAIN pairs", "audit_pairs_median.png")

In [ ]:
summary = dict(n_images=int(len(df)), n_aug_train=int(is_tr.sum()), n_aug_test=int((~is_tr).sum()),
               nn_cos_median=float(np.median(nn_sim)), one_nn_label_agreement=float(same_cls.mean()),
               groups=group_summary)
json.dump(summary, open(f"{OUT}/audit_summary.json", "w"), indent=2)
print("SUMMARY\n" + json.dumps(summary, indent=2))
print("\nSend back:", sorted(f for f in os.listdir(OUT) if f.startswith("audit_") and not f.endswith(".npy")))